# Detecção de Anomalias em Transações Financeiras



Este projeto aplica técnicas de detecção de anomalias para identificar possíveis fraudes em transações de cartão de crédito, utilizando o algoritmo **Isolation Forest**.



O dataset utilizado é o [Credit Card Fraud Detection](https://www.kaggle.com/mlg-ulb/creditcardfraud), disponibilizado pela ULB (Université Libre de Bruxelles), contendo transações realizadas por portadores de cartão europeus em setembro de 2013.



Por questões de confidencialidade, as features `V1` a `V28` já são resultado de uma transformação PCA aplicada aos dados originais. Apenas `Time` e `Amount` mantêm seus valores originais.



**Objetivo:** treinar um modelo não supervisionado capaz de sinalizar transações anômalas sem depender dos rótulos de fraude durante o treinamento, e avaliar seu desempenho comparando com os rótulos reais.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Carregamento dos dados

In [ ]:
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

dados = pd.read_csv(path + "/creditcard.csv")

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
dados.info()

In [ ]:
dados["Class"].value_counts()

O dataset possui 284.807 transações financeiras e 31 atributos. A variável `Class` representa a classificação da transação, sendo 0 para transações legítimas e 1 para fraudes. Observa-se um forte desbalanceamento entre as classes, com uma quantidade muito reduzida de registros fraudulentos.

## 2. Análise exploratória

In [ ]:
print("Valores nulos por coluna:")
print(dados.isnull().sum())

In [ ]:
print("\nEstatísticas gerais:")
print(dados.describe())

In [ ]:
print("\nQuantidade de transações por classe:")
print(dados["Class"].value_counts())

print("\nPorcentagem de cada classe:")
print((dados["Class"].value_counts(normalize=True) * 100).round(3))

In [ ]:
print("\nResumo da coluna Amount:")
print(dados["Amount"].describe())

In [ ]:
dadosFraudes = dados[dados["Class"] == 1]
dadosNormais = dados[dados["Class"] == 0]

print("\nResumo das transações fraudulentas:")
print(dadosFraudes["Amount"].describe())

print("\nResumo das transações normais:")
print(dadosNormais["Amount"].describe())

In [ ]:
plt.figure(figsize=(10,6))

plt.hist(
    dados["Amount"],
    bins=50,
    edgecolor="black",
    alpha=0.7
)

plt.axvline(
    dados["Amount"].mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Média = {dados['Amount'].mean():.2f}"
)

plt.title("Distribuição dos valores das transações")
plt.xlabel("Valor da transação")
plt.ylabel("Quantidade")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(10,4))

plt.boxplot(
    dados["Amount"],
    vert=False
)

plt.title("Boxplot dos valores das transações")
plt.xlabel("Valor")

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

dados["Class"].value_counts().plot(
    kind="bar"
)

plt.title("Quantidade de transações por classe")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.xticks([0,1],["Normal","Fraude"], rotation=0)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.boxplot(
    [
        dadosNormais["Amount"],
        dadosFraudes["Amount"]
    ],
    tick_labels=["Normal","Fraude"]
)

plt.title("Comparação dos valores das transações")
plt.ylabel("Valor")

plt.show()

### 2.1 Transações ao longo do tempo

A coluna `Time` representa os segundos decorridos desde a primeira transação do dataset. Convertendo para hora do dia é possível verificar se fraudes se concentram em algum horário.

In [ ]:
dados["Hora"] = (dados["Time"] // 3600) % 24

plt.figure(figsize=(10,5))

sns.histplot(
    data=dados,
    x="Hora",
    hue="Class",
    bins=24,
    stat="density",
    common_norm=False,
    palette={0: "steelblue", 1: "crimson"}
)

plt.title("Distribuição de transações por hora do dia (normalizada por classe)")
plt.xlabel("Hora do dia")
plt.ylabel("Densidade")

plt.show()

### 2.2 Correlação das variáveis com a classe

Como `V1` a `V28` já são componentes de uma PCA, elas são naturalmente pouco correlacionadas entre si. Ainda assim, vale observar quais componentes mais se relacionam com a ocorrência de fraude.

In [ ]:
correlacoes = dados.drop(columns=["Hora"]).corr()["Class"].drop("Class").sort_values()

plt.figure(figsize=(8,10))

correlacoes.plot(kind="barh", color="slateblue")

plt.title("Correlação de cada variável com a classe (Fraude)")
plt.xlabel("Coeficiente de correlação")

plt.show()

**Conclusão da análise exploratória**

O conjunto de dados possui 284.807 transações financeiras e 31 atributos.
Não foram encontrados valores nulos.
O dataset é altamente desbalanceado, contendo aproximadamente 99,83% de transações legítimas e apenas 0,17% de fraudes.
A variável `Amount` apresenta distribuição assimétrica, com predominância de transações de baixo valor e presença de valores extremos (outliers).
As estatísticas indicam diferenças entre os valores das transações normais e fraudulentas, e algumas componentes da PCA (como V17, V14, V12 e V10) mostram correlação mais forte com a classe fraude, justificando a aplicação de um algoritmo de detecção de anomalias.

## 3. Pré-processamento

As variáveis `V1` a `V28` já saem da PCA em escala comparável entre si. As colunas que realmente precisam de padronização são `Time` e `Amount`, então o `StandardScaler` é aplicado apenas a elas. A coluna auxiliar `Hora` (criada só para a análise exploratória) é descartada do conjunto de treino.

In [ ]:
X = dados.drop(columns=["Class", "Hora"])
y = dados["Class"]

colunas_para_escalar = ["Time", "Amount"]

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[colunas_para_escalar] = scaler.fit_transform(X[colunas_para_escalar])

### 3.1 Separação em treino e teste

Mesmo sendo um modelo não supervisionado, treinar e avaliar sobre o mesmo conjunto de dados infla a percepção de desempenho, já que o modelo "viu" exatamente os pontos que está sendo avaliado. Para simular um cenário mais realista, o modelo é treinado em uma parte dos dados e avaliado em uma parte que ele nunca viu.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X_scaled,
    y,
    test_size=0.3,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Treino:", X_treino.shape, "| Teste:", X_teste.shape)
print("Proporção de fraudes no treino:", y_treino.mean().round(5))
print("Proporção de fraudes no teste:", y_teste.mean().round(5))

## 4. Treinamento do modelo

O parâmetro `contamination` foi definido a partir da proporção real de fraudes observada no treino. Vale destacar que, em um cenário de produção, essa proporção geralmente não é conhecida de antemão; nesse caso, o valor foi usado como aproximação inicial, mas o ideal é também observar a curva de precision-recall para escolher um limiar de anomalia mais robusto (feito na seção 5).

In [ ]:
taxa_fraude_treino = y_treino.mean()

modelo = IsolationForest(
    n_estimators=200,
    contamination=taxa_fraude_treino,
    max_samples="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

modelo.fit(X_treino)

In [ ]:
predicoes = modelo.predict(X_teste)
predicoes = np.where(predicoes == -1, 1, 0)

scores_anomalia = -modelo.score_samples(X_teste)  # quanto maior, mais anômalo

In [ ]:
print("Anomalias detectadas no conjunto de teste:")
print(pd.Series(predicoes).value_counts())

## 5. Avaliação do modelo

In [ ]:
matriz = confusion_matrix(y_teste, predicoes)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Normal", "Fraude"]
)

disp.plot(cmap="Blues")

plt.title("Matriz de Confusão (conjunto de teste)")

plt.show()

In [ ]:
print(classification_report(y_teste, predicoes))

### 5.1 ROC-AUC e Average Precision

Em problemas tão desbalanceados, métricas baseadas apenas na predição binária (0/1) escondem informação. O score contínuo de anomalia permite calcular ROC-AUC e Average Precision (área sob a curva precision-recall), que avaliam a qualidade da ordenação das transações por risco, não apenas o corte final.

In [ ]:
roc_auc = roc_auc_score(y_teste, scores_anomalia)
avg_precision = average_precision_score(y_teste, scores_anomalia)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Average Precision (PR-AUC): {avg_precision:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

RocCurveDisplay.from_predictions(y_teste, scores_anomalia, ax=axes[0])
axes[0].set_title("Curva ROC")

PrecisionRecallDisplay.from_predictions(y_teste, scores_anomalia, ax=axes[1])
axes[1].set_title("Curva Precision-Recall")

plt.tight_layout()
plt.show()

### 5.2 Distribuição dos scores de anomalia por classe

Se o modelo estiver separando bem os dois grupos, o score de anomalia das fraudes deve se concentrar em valores mais altos do que o das transações normais.

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    x=scores_anomalia,
    hue=y_teste.map({0: "Normal", 1: "Fraude"}),
    bins=50,
    stat="density",
    common_norm=False,
    palette={"Normal": "steelblue", "Fraude": "crimson"}
)

plt.title("Distribuição do score de anomalia por classe")
plt.xlabel("Score de anomalia (quanto maior, mais anômalo)")
plt.ylabel("Densidade")

plt.show()

In [ ]:
anomalias = X_teste[predicoes == 1].copy()
normais = X_teste[predicoes == 0].copy()

# recupera o Amount original (antes da padronização) para interpretação
anomalias_amount = dados.loc[X_teste.index][predicoes == 1]["Amount"]
normais_amount = dados.loc[X_teste.index][predicoes == 0]["Amount"]

plt.figure(figsize=(10,5))

plt.boxplot(
    [normais_amount, anomalias_amount],
    tick_labels=["Normal","Anomalia"]
)

plt.title("Comparação dos valores das transações detectadas")
plt.ylabel("Valor")

plt.show()

## 6. Salvando o modelo

Para reaproveitar o modelo treinado sem precisar refazer o treinamento (por exemplo, em uma API ou pipeline de scoring), ele é serializado junto com o scaler.

In [ ]:
joblib.dump(modelo, "isolation_forest_fraude.joblib")
joblib.dump(scaler, "scaler_fraude.joblib")

print("Modelo e scaler salvos com sucesso.")

## 7. Conclusão

Neste projeto foi desenvolvido um modelo de detecção de anomalias utilizando o algoritmo Isolation Forest para identificar possíveis fraudes em transações financeiras.

Após a análise exploratória dos dados, verificou-se que o conjunto de dados é altamente desbalanceado, contendo aproximadamente 99,83% de transações legítimas. Algumas componentes da PCA (V17, V14, V12, V10) apresentaram correlação mais forte com a ocorrência de fraude.

O modelo foi treinado de forma não supervisionada, sem utilizar a variável alvo durante o treinamento, e avaliado em um conjunto de teste separado para simular um cenário mais próximo do real. Além da matriz de confusão e do classification report, foram calculadas métricas mais adequadas a problemas desbalanceados (ROC-AUC e Average Precision), que avaliam a qualidade do ranking de risco produzido pelo modelo, e não apenas o corte binário final.

Embora o Isolation Forest consiga identificar parte das transações fraudulentas, ainda ocorrem falsos positivos e fraudes não detectadas, demonstrando que a detecção de anomalias é um problema complexo. Possíveis melhorias incluem ajuste fino de hiperparâmetros, engenharia de atributos adicional, comparação com outros algoritmos (Local Outlier Factor, One-Class SVM, Autoencoders) e o uso de um limiar de decisão calibrado a partir da curva precision-recall em vez de assumir a taxa real de fraude como contamination.